# LSTM的结构与代码

来自视频 [徒手实现长短期记忆网络](https://www.bilibili.com/video/BV1pE421P7Ej) 。

LSTM（长短期记忆网络）是一种特殊的循环神经网络（RNN），它能够学习长期依赖信息。LSTM的关键在于其内部结构，它包含了三个门（输入门、遗忘门、输出门）和一个细胞状态（cell state），这些结构使得LSTM能够在处理序列数据时，有效地保留长期信息和忽略无关信息。

LSTM的核心结构包括：

+ 输入门：决定哪些信息需要更新。
+ 遗忘门：决定哪些信息需要丢弃。
+ 输出门：决定哪些信息需要输出。
+ 细胞状态：作为信息的传递载体，保留重要的长期信息。

在传统的 RNN 网络中， $h_t$ 既作为输出又作为隐藏状态传递，

而在 LSTM 中， $C_t$ （记忆单元）作为主要的记忆载体，而 $h_t$ 是 $C_t$ 中“节选”出来用于输出和传递的部分。

相较于 RNN ，LSTM 中的同一个时刻的计算需要用到：当前输入 $x_t$ ，上一时刻隐藏状态 $h_{t-1}$ 和上一时刻细胞状态 $C_{t-1}$，并计算的结构有这个时刻的隐藏状态和这个时刻的细胞状态，隐藏状态作为语言建模头。

另外，上述的隐藏状态和细胞状态的计算都有此时刻的输入和上一时刻的输出参与，由门控控制。

下面是 RNN 的短期记忆的几张图：

![](./images/RNN%20短期记忆1.png)
![](./images/RNN%20短期记忆2.png)


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
import matplotlib.pyplot as plt
%matplotlib inline


torch.manual_seed(12046)

In [3]:
# 一些超参数
learning_rate = 1e-3
eval_iters = 10
batch_size=1000
sequence_len=64
# 如果有GPU，该脚本将使用GPU进行计算
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
datasets = load_dataset('json', data_files='./datasets/python/final/jsonl/train/*.jsonl.gz')
datasets = datasets['train'].filter(lambda x: 'apache/spark' in x['repo'])

In [5]:
class CharTokenizer:

    def __init__(self, data, end_ind=0):
        # data: list[str]
        # 得到所有的字符
        chars = sorted(list(set(''.join(data))))
        self.char2ind = {s: i + 1 for i, s in enumerate(chars)}
        self.char2ind['<|e|>'] = end_ind
        self.ind2char = {v: k for k, v in self.char2ind.items()}
        self.end_ind = end_ind

    def encode(self, x):
        # x: str
        return [self.char2ind[i] for i in x]

    def decode(self, x):
        # x: int or list[x]
        if isinstance(x, int):
            return self.ind2char[x]
        return [self.ind2char[i] for i in x]

tokenizer = CharTokenizer(datasets['original_string'])
test_str = 'def f(x):'
re = tokenizer.encode(test_str)
print(re)
''.join(tokenizer.decode(range(len(tokenizer.char2ind))))

[70, 71, 72, 2, 72, 10, 90, 11, 28]


'<|e|>\n !"#$%&\'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\\]^_`abcdefghijklmnopqrstuvwxyz{|}~ö'

In [11]:
def process(data, tokenizer, sequence_len=sequence_len):
    text = data['original_string']
    # text is list[str]
    inputs, labels = [], []
    for t in text:
        enc = tokenizer.encode(t)
        enc += [tokenizer.end_ind]
        # 有bug，无法处理长度过小的数据
        for i in range(len(enc) - sequence_len):
            inputs.append(enc[i: i + sequence_len])
            labels.append(enc[i + 1: i + 1 + sequence_len])
    return {'inputs': inputs, 'labels': labels}

# 将数据分为训练集和测试集
tokenized = datasets.train_test_split(test_size=0.1, seed=1024, shuffle=True)

f = lambda x: process(x, tokenizer)
tokenized = tokenized.map(f, batched=True, remove_columns=datasets.column_names)
tokenized.set_format(type='torch', device=device)


In [12]:
train_loader = DataLoader(tokenized['train'], batch_size=batch_size, shuffle=True)
test_loader = DataLoader(tokenized['test'], batch_size=batch_size, shuffle=True)
next(iter(train_loader))

{'inputs': tensor([[71, 94,  2,  ...,  2,  2,  2],
         [70, 85, 86,  ...,  4,  4,  4],
         [86, 74,  2,  ..., 81, 87, 82],
         ...,
         [82, 71, 10,  ..., 54, 91, 82],
         [71, 69, 81,  ...,  2,  2,  2],
         [84, 75, 80,  ..., 86,  2, 75]], device='cuda:0'),
 'labels': tensor([[94,  2,  2,  ...,  2,  2,  2],
         [85, 86, 84,  ...,  4,  4,  1],
         [74,  2, 80,  ..., 87, 82, 75],
         ...,
         [71, 10, 21,  ..., 91, 82, 71],
         [69, 81, 84,  ...,  2,  2,  2],
         [75, 80, 73,  ...,  2, 75, 85]], device='cuda:0')}

In [13]:
@torch.no_grad()
def generate(model, context, tokenizer, max_new_tokens=300):
    # context: (1, T)
    #out = []
    out = context.tolist()[0]
    model.eval()
    for _ in range(max_new_tokens):
        #可以考虑截断背景，使得文本生成更加贴近训练
        #logits = model(context[:, -sequence_len:])
        logits = model(context)            # (1, T, 98)
        probs = F.softmax(logits[:, -1, :], dim=-1)  # (1, 98)
        # 随机生成文本
        ix = torch.multinomial(probs, num_samples=1)  # (1, 1)
        # 更新背景
        context = torch.concat((context, ix), dim=-1)
        out.append(ix.item())
        if out[-1] == tokenizer.end_ind:
            break
    model.train()
    return out

In [15]:
def estimate_loss(model):
    re = {}
    # 将模型切换至评估模式
    model.eval()
    re['train'] = _loss(model, train_loader)
    re['test'] = _loss(model, test_loader)
    # 将模型切换至训练模式
    model.train()
    return re

@torch.no_grad()
def _loss(model, data_loader):
    """
    计算模型在不同数据集下面的评估指标
    """
    loss = []
    data_iter= iter(data_loader)
    # 随机使用多个批量数据来预估模型效果
    for k in range(eval_iters):
        data = next(data_iter, None)
        if data is None:
            data_iter = iter(data_loader)
            data = next(data_iter, None)
        inputs, labels = data['inputs'], data['labels']  # (B, T)
        logits = model(inputs)                           # (B, T, vs)
        # 请参考官方文档
        loss.append(F.cross_entropy(logits.transpose(-2, -1), labels).item())
    return torch.tensor(loss).mean().item()

In [17]:
def train_model(model, optimizer, epochs=10):
    # 记录模型在训练集上的模型损失
    lossi = []
    for epoch in range(epochs):
        for i, data in enumerate(train_loader, 0):
            inputs, labels = data['inputs'], data['labels']  # (B, T)
            optimizer.zero_grad()
            logits = model(inputs)                           # (B, T, vs)
            loss = F.cross_entropy(logits.transpose(-2, -1), labels)
            lossi.append(loss.item())
            loss.backward()
            optimizer.step()
        # 评估模型，并输出结果
        stats = estimate_loss(model)
        train_loss = f'train loss {stats["train"]:.4f}'
        test_loss = f'test loss {stats["test"]:.4f}'
        print(f'epoch {epoch:>2}: {train_loss}, {test_loss}')
    return lossi

![](./images/LSTM.png)

In [18]:
class LSTMCell(nn.Module):

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        combined_size = input_size + hidden_size

        self.forget_gate = nn.Linear(combined_size, hidden_size)
        self.in_gate = nn.Linear(combined_size, hidden_size)
        self.new_cell_state = nn.Linear(combined_size, hidden_size)
        self.out_gate = nn.Linear(combined_size, hidden_size)

    def forward(self, input, state=None):
        # input: (B, I)
        # state: ((B, H), (B, H))
        B = input.shape[0]
        if state is None:
            state = self.init_state(B, input.device)
        hs, cs = state
        combined = torch.concat((input, hs), dim=-1)  # (B, I + H)

        # 细胞状态更新
        ingate = F.sigmoid(self.in_gate(combined))
        forgetgate = F.sigmoid(self.forget_gate(combined))
        ncs = F.tanh(self.new_cell_state(combined))

        cs = cs * forgetgate + ingate * ncs

        # 隐藏状态更新
        outgate = F.sigmoid(self.out_gate(combined))
        hs = F.tanh(cs) * outgate
        return hs, cs

    def init_state(self, B, device):
        hs = torch.zeros((B, self.hidden_size), device=device)
        cs = torch.zeros((B, self.hidden_size), device=device)
        return hs, cs

In [19]:
l_cell = LSTMCell(3, 4)
x = torch.randn(5, 3)
a, b = l_cell(x)
a.shape, b.shape

(torch.Size([5, 4]), torch.Size([5, 4]))

In [20]:
class LSTM(nn.Module):

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.cell = LSTMCell(input_size, hidden_size)

    def forward(self, input, state=None):
        # input:  (B, T, C)
        # state:  ((B, H), (B, H))
        # out:    (B, T, H)
        B, T, C = input.shape
        re = []
        for i in range(T):
            state = self.cell(input[:, i, :], state)
            re.append(state[0])
        return torch.stack(re, dim=1)                                  # (B, T, H)

In [22]:
def test_lstm():
    '''
    测试LSTM实现的准确性
    '''
    # 随机生成模型结构
    B, T, input_size, hidden_size, num_layers = torch.randint(1, 20, (5,)).tolist()
    ref_model = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True)
    # 随机生成输入
    inputs = torch.randn(B, T, input_size)
    hs, cs = torch.randn((2 * num_layers, B, hidden_size)).chunk(2, 0)
    re = inputs
    # 取出模型参数
    for layer_index in range(num_layers):
        l = ref_model.all_weights[layer_index]
        if layer_index == 0:
            model = LSTM(input_size, hidden_size)
        else:
            model = LSTM(hidden_size, hidden_size)
        i, f, c, o = torch.cat((l[0], l[1]), dim=1).chunk(4, 0)
        ib, fb, cb, ob = (l[2] + l[3]).chunk(4, 0)
        # 设置模型参数
        model.cell.in_gate.weight = nn.Parameter(i)
        model.cell.in_gate.bias = nn.Parameter(ib)
        model.cell.forget_gate.weight = nn.Parameter(f)
        model.cell.forget_gate.bias = nn.Parameter(fb)
        model.cell.new_cell_state.weight = nn.Parameter(c)
        model.cell.new_cell_state.bias = nn.Parameter(cb)
        model.cell.out_gate.weight = nn.Parameter(o)
        model.cell.out_gate.bias = nn.Parameter(ob)
        # 计算隐藏状态
        re = model(re, (hs[layer_index], cs[layer_index]))
    ref_re, _ = ref_model(inputs, (hs, cs))
    # 验证计算结果（最后一层的隐藏状态是否一致）
    out = torch.all(torch.abs(re - ref_re) < 1e-4)
    return out, (B, T, input_size, hidden_size, num_layers)

test_lstm()

(tensor(True), (12, 8, 15, 11, 6))

In [27]:
class CharLSTM(nn.Module):  # 语言建模

    def __init__(self, vs):
        super().__init__()

        self.emb_size = 256
        self.hidden_size = 128
        self.emb = nn.Embedding(vs, self.emb_size)
        self.dp = nn.Dropout(0.4)
        self.lstm1 = LSTM(self.emb_size, self.hidden_size)
        self.ln1 = nn.LayerNorm(self.hidden_size)  # 包含两个可学习参数：gamma (weight)：缩放 beta (bias)：平移
        self.lstm2 = LSTM(self.hidden_size, self.hidden_size)
        self.ln2 = nn.LayerNorm(self.hidden_size)
        self.lstm3 = LSTM(self.hidden_size, self.hidden_size)
        self.ln3 = nn.LayerNorm(self.hidden_size)
        self.lm = nn.Linear(self.hidden_size, vs)


    def forward(self, x):
        # x: (B, T)
        embeddings = self.emb(x)
        h = self.ln1(self.dp(self.lstm1(embeddings)))  # (B, T, H)
        h = self.ln2(self.dp(self.lstm2(h)))
        h = self.ln3(self.dp(self.lstm3(h)))
        output = self.lm(h)
        return output

In [28]:
c_model = CharLSTM(len(tokenizer.char2ind)).to('cuda')
c_model

CharLSTM(
  (emb): Embedding(98, 256)
  (dp): Dropout(p=0.4, inplace=False)
  (lstm1): LSTM(
    (cell): LSTMCell(
      (forget_gate): Linear(in_features=384, out_features=128, bias=True)
      (in_gate): Linear(in_features=384, out_features=128, bias=True)
      (new_cell_state): Linear(in_features=384, out_features=128, bias=True)
      (out_gate): Linear(in_features=384, out_features=128, bias=True)
    )
  )
  (ln1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (lstm2): LSTM(
    (cell): LSTMCell(
      (forget_gate): Linear(in_features=256, out_features=128, bias=True)
      (in_gate): Linear(in_features=256, out_features=128, bias=True)
      (new_cell_state): Linear(in_features=256, out_features=128, bias=True)
      (out_gate): Linear(in_features=256, out_features=128, bias=True)
    )
  )
  (ln2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (lstm3): LSTM(
    (cell): LSTMCell(
      (forget_gate): Linear(in_features=256, out_features=128, bias=True)
   

In [29]:
context = torch.tensor(tokenizer.encode('def'), device=device).unsqueeze(0)
print(''.join(tokenizer.decode(generate(c_model, context, tokenizer))))

def*ZKV
oo|("YA{ByE G|uw=3<1'L$?Q9NN[{/Q=CK|AM:iKcam;+Q3m<sA!gW`$ö8Nx!q9T3"yMm5Za)'c~5\rm&B"T{r
c"tM=^Dax1z#<|e|>


In [30]:
estimate_loss(c_model)

{'train': 4.643476486206055, 'test': 4.641425132751465}

In [31]:
l = train_model(c_model, optim.Adam(c_model.parameters(), lr=learning_rate))

epoch  0: train loss 1.2523, test loss 1.4218
epoch  1: train loss 1.1358, test loss 1.3298
epoch  2: train loss 1.0366, test loss 1.2401
epoch  3: train loss 0.9966, test loss 1.2153
epoch  4: train loss 0.9600, test loss 1.2028
epoch  5: train loss 0.9417, test loss 1.1896
epoch  6: train loss 0.9280, test loss 1.1952
epoch  7: train loss 0.9188, test loss 1.1848
epoch  8: train loss 0.8910, test loss 1.1795
epoch  9: train loss 0.8898, test loss 1.1726


In [32]:
context = torch.tensor(tokenizer.encode('def'), device=device).unsqueeze(0)
print(''.join(tokenizer.decode(generate(c_model, context, tokenizer))))

def quote=Value, size=None, partitionFunc=None, UDFType=None, options=None):
        """
        Create a DataFrame is not None:
            if not self._sc.startSize():
                return self._jdf.sample()
        jrdd = self.mapPartitionsWithIndex()
                return starts[0] = np.memory.a
